# 📊 Notebook 01 — EDA & Data Cleaning
Run cells top-to-bottom after placing datasets in `data/raw/`

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sys.path.append(os.path.join(os.getcwd(), '..'))
from src.text_cleaner import basic_clean, advanced_clean

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
print('✅ Imports OK')

In [ ]:
df = pd.read_csv('../data/raw/resumes.csv')
df.columns = [c.strip().lower() for c in df.columns]
print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head(3)

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('Duplicates:', df.duplicated().sum())
print('Categories:', df['category'].nunique())

In [ ]:
df = df.drop_duplicates().dropna(subset=['category','resume'])
df['category'] = df['category'].str.strip()
print('Cleaning text (60s)...')
df['resume_clean_basic']    = df['resume'].apply(basic_clean)
df['resume_clean_advanced'] = df['resume'].apply(advanced_clean)
df['resume_word_count']     = df['resume_clean_basic'].apply(lambda x: len(str(x).split()))
df.to_csv('../data/processed/cleaned_resumes.csv', index=False)
print(f'✅ Saved {len(df)} rows')

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(16,6))
counts = df['category'].value_counts()
axes[0].barh(counts.index, counts.values)
axes[0].set_title('Resume Count by Category')
axes[0].set_xlabel('Count')
top8 = counts.head(8)
others = counts.iloc[8:].sum()
pie = pd.concat([top8, pd.Series({'Others':others})])
axes[1].pie(pie.values, labels=pie.index, autopct='%1.1f%%', startangle=140)
axes[1].set_title('Category Share')
plt.tight_layout()
plt.savefig('../screenshots/plot1_categories.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].hist(df['resume_word_count'], bins=40, color='#667eea', edgecolor='white')
axes[0].axvline(df['resume_word_count'].mean(), color='red', linestyle='--',
                label=f"Mean: {df['resume_word_count'].mean():.0f}")
axes[0].set_title('Word Count Distribution')
axes[0].legend()
mean_wc = df.groupby('category')['resume_word_count'].mean().sort_values()
axes[1].barh(mean_wc.index, mean_wc.values, color='#764ba2')
axes[1].set_title('Avg Word Count per Category')
plt.tight_layout()
plt.savefig('../screenshots/plot2_wordcount.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
all_text = ' '.join(df['resume_clean_advanced'].dropna())
freq = Counter(all_text.split())
top30 = pd.DataFrame(freq.most_common(30), columns=['word','count'])
plt.figure(figsize=(12,6))
plt.barh(top30['word'][::-1], top30['count'][::-1], color='#667eea')
plt.title('Top 30 Keywords in Resumes')
plt.xlabel('Frequency')
plt.tight_layout()
plt.savefig('../screenshots/plot3_keywords.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ EDA complete! Check screenshots/ folder')